Week 5 - Spark Data Cleaning, Transformation & Aggregation

Name: Unnati Agarwal
Internship: Celebal Technologies
Week: 5

In [1]:
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week5Assignment") \
    .getOrCreate()

In [5]:
from google.colab import files

uploaded = files.upload()

Saving sample_sales.csv to sample_sales.csv


In [6]:
df = spark.read.csv(
    "sample_sales.csv",
    header=True,
    inferSchema=True
)

df.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   status|age|subscription|city|     email|username|price|store_id|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    102|      2024-01-06|  East|       Furniture|      320.0|     NULL| 34|       Basic|  NY|b@mail.com|     bob|320.0|    S102|2024-01-06 11:20:00|
|    103|      2024-01-07|  West|     Electronics|      610.0|Completed| 29|     Premium|  SF|      

Q1: Key limitations of MapReduce that make Spark preferred


Disk I/O overhead — MapReduce writes intermediate results to disk (HDFS) after every Map/Reduce phase, causing heavy latency across multi-stage jobs.
Poor support for iterative algorithms — ML/graph algorithms that loop over the same data re-read from disk every iteration.
Batch-only processing — no native streaming support; Spark handles both batch and streaming.
Rigid programming model — only Map/Reduce steps exist; complex pipelines need many chained jobs.
High latency — new JVM per task, no in-memory caching, slow for interactive queries.
Limited APIs — mostly Java; Spark supports Python, Scala, Java, R, SQL, plus MLlib/GraphX.




Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.
Answer

Spark stores intermediate data in RAM instead of writing it to disk after each operation.

In iterative machine learning algorithms (such as K-Means or Gradient Descent), the same dataset is processed multiple times.

Because Spark caches data in memory:

Data is loaded only once.
Repeated disk reads are avoided.
CPU utilization increases.
Training time is significantly reduced.

MapReduce reloads data from disk during every iteration, making it much slower.

Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [30]:


df_clean = df.dropDuplicates(["user_id", "transaction_date"])

df_clean.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   status|age|subscription|city|     email|username|price|store_id|         event_time|
+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    102|      2024-01-06|  East|       Furniture|      320.0|     NULL| 34|       Basic|  NY|b@mail.com|     bob|320.0|    S102|2024-01-06 11:20:00|
|    103|      2024-01-07|  West|     Electronics|      610.0|Completed| 29|     Premium|  SF|      NULL|   carol|610.0|    S101|2024-01-07 09:05:00|
|    104|      2024-01-08|  West|         Apparel|       85.0|  Pending| 41|       Basic|  LA|d@mail

Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [12]:
from pyspark.sql.functions import avg

df.filter(df.region == "West") \
.groupBy("product_category") \
.agg(avg("sale_amount").alias("Average_Sales")) \
.show()

+----------------+-------------+
|product_category|Average_Sales|
+----------------+-------------+
|         Apparel|         90.0|
|     Electronics|        504.0|
|       Furniture|        260.0|
+----------------+-------------+



Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.


ANS - .na.drop() removes rows or columns containing missing values (NaN or null) from your dataset. In contrast, .na.fill() replaces those missing values with a specified substitute, such as a number, string, or statistical value.

In [13]:
df.na.fill({"status": "Unknown"}).show()

+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   status|age|subscription|city|     email|username|price|store_id|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    102|      2024-01-06|  East|       Furniture|      320.0|  Unknown| 34|       Basic|  NY|b@mail.com|     bob|320.0|    S102|2024-01-06 11:20:00|
|    103|      2024-01-07|  West|     Electronics|      610.0|Completed| 29|     Premium|  SF|      

Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [14]:
from pyspark.sql.functions import count

df.groupBy("city") \
.agg(count("*").alias("Total")) \
.filter("Total > 100") \
.show()

+----+-----+
|city|Total|
+----+-----+
+----+-----+



In [15]:
df.groupBy("city") \
.agg(count("*").alias("Total")) \
.show()

+----+-----+
|city|Total|
+----+-----+
|  LA|    3|
|  SF|    6|
|  NY|    3|
| HOU|    3|
+----+-----+



Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

ANS- Spark DataFrames are immutable — every cleaning operation returns a new DataFrame rather than modifying the original. Because nothing is mutated in place, the original DataFrame stays valid for reuse/debugging, and Spark's Catalyst optimizer can rewrite the full chain of transformations before executing anything (lazy evaluation). It also underpins fault tolerance — lost partitions can be recomputed from lineage.

Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [16]:
df.filter(
    (df.age >= 18) &
    (df.age <= 30) &
    (df.subscription == "Premium")
).show()

+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   status|age|subscription|city|     email|username|price|store_id|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    103|      2024-01-07|  West|     Electronics|      610.0|Completed| 29|     Premium|  SF|      NULL|   carol|610.0|    S101|2024-01-07 09:05:00|
|    105|      2024-01-08| South|     Electronics|      275.0|Completed| 18|     Premium| HOU|e@mail

Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

ANS - sum()/avg() skip nulls silently by default — avg() divides by the count of non-null values, not total rows, which can distort the average if nulls actually represent zeros.
If nulls mean "no purchase = 0" rather than "unknown," leaving them null undercounts records in count()/avg().
Unhandled nulls can break joins, UDFs, or ML feature vectors downstream.
Explicitly deciding to drop vs. fill nulls before aggregating makes the result's meaning unambiguous and reproducible, rather than depending on Spark's default null-skipping behavior.

Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [17]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

df = df.withColumn(
    "event_time",
    col("raw_timestamp").cast(TimestampType())
).drop("raw_timestamp")

df.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   status|age|subscription|city|     email|username|price|store_id|         event_time|
+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    102|      2024-01-06|  East|       Furniture|      320.0|     NULL| 34|       Basic|  NY|b@mail.com|     bob|320.0|    S102|2024-01-06 11:20:00|
|    103|      2024-01-07|  West|     Electronics|      610.0|Completed| 29|     Premium|  SF|      

Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

ANS- Shuffle: groupBy requires records with the same key to be physically brought together, but they're usually scattered across different partitions/nodes. Spark:


Redistributes (shuffles) data across the cluster, keyed by the grouping column.
Transfers that data over the network to the executor owning each key's partition.
Merges/aggregates the shuffled data on the receiving side.


Narrow transformation (filter, select, withColumn): output partition depends only on one input partition — no data movement.
Wide transformation (groupBy, join, distinct, orderBy): output partition may depend on data scattered across all input partitions, forcing a shuffle. Shuffles are usually the biggest performance bottleneck in Spark — minimizing them (broadcast joins, pre-aggregation, sensible partitioning) is a core tuning strategy.

Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [18]:
from pyspark.sql.functions import col

df.filter(
    col("email").isNotNull() &
    (col("username") != "")
).show()

+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   status|age|subscription|city|     email|username|price|store_id|         event_time|
+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    102|      2024-01-06|  East|       Furniture|      320.0|     NULL| 34|       Basic|  NY|b@mail.com|     bob|320.0|    S102|2024-01-06 11:20:00|
|    105|      2024-01-08| South|     Electronics|      275.0|Completed| 18|     Premium| HOU|e@mail

Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [19]:
from pyspark.sql.functions import min, max, avg

df.agg(
    min("price").alias("Minimum"),
    max("price").alias("Maximum"),
    avg("price").alias("Average")
).show()

+-------+-------+------------------+
|Minimum|Maximum|           Average|
+-------+-------+------------------+
|   60.0|  610.0|313.84615384615387|
+-------+-------+------------------+



Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

ANS- Spark may fall back to StringType if formats are inconsistent, silently disabling date operations.
Values that don't match the inferred format become null without any warning — quiet data loss.
Ambiguous formats (03/04/2024) parse inconsistently depending on assumed locale.
Inference requires an extra pass over the data, adding overhead on large files.
Results can vary between runs if the sampled rows used for inference differ — non-reproducible.

Q15: Write a final processing pipeline that:

Filters out duplicates.

Fills null prices with 0.

Groups by store_id to calculate total revenue.



In [31]:
from pyspark.sql.functions import sum

final_df = (
    df
    .dropDuplicates()
    .na.fill({"price": 0})
    .groupBy("store_id")
    .agg(sum("price").alias("Total_Revenue"))
)

final_df.show()

+--------+-------------+
|store_id|Total_Revenue|
+--------+-------------+
|    S102|        860.0|
|    S104|       1095.0|
|    S101|       1675.0|
+--------+-------------+

